In [4]:
# === Few-shot (training-free) baseline: Nearest Prototype on embeddings ===

In [1]:
!nvidia-smi

Sat Nov  1 20:39:04 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score
from sklearn.preprocessing import normalize

In [7]:
# ---- 1) Load participants & embeddings ----
participants_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Autism/participants.csv')
X_participants = participants_df["ParticipantID"].astype(str).values
y_participants = participants_df["Label"].astype(int).values  # 0/1 label per participant

ds = np.load('/content/drive/MyDrive/Colab Notebooks/Autism/Few-Shot/ASD_vgg16.npz', allow_pickle=True)
X   = ds['X'].astype('float32')   # embeddings
y   = ds['y'].astype('int32')     # 0=TD, 1=ASD
IDs = ds['IDs'].astype(str)       # per-image participant ID

In [8]:
# L2-normalize embeddings for cosine similarity
Z = normalize(X, norm='l2', axis=1)

print("Embeddings:", Z.shape, "| images:", len(y), "| participants:", len(np.unique(IDs)))

# ---- 2) CV setup (participant-wise) ----
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

Embeddings: (3282, 512) | images: 3282 | participants: 54


In [9]:
# K-shot settings
K_list = [1, 3, 5, 10]
repeats = 30  # repeat random support sampling for stability
temperature = 0.2  # softmax temperature for converting cosine sims to probs

def few_shot_fold_metrics(Z_train, y_train, Z_test, y_test, K):
    # indices by class in TRAIN set
    tr0 = np.where(y_train == 0)[0]
    tr1 = np.where(y_train == 1)[0]
    if len(tr0) == 0 or len(tr1) == 0:
        return None  # can't form supports

    logits_accum = np.zeros((len(Z_test), 2), dtype='float32')

    for _ in range(repeats):
        # sample K supports per class (with replacement if needed)
        s0 = np.random.choice(tr0, size=K, replace=(K > len(tr0)))
        s1 = np.random.choice(tr1, size=K, replace=(K > len(tr1)))

        P0 = Z_train[s0].mean(axis=0, keepdims=True)  # TD prototype
        P1 = Z_train[s1].mean(axis=0, keepdims=True)  # ASD prototype
        # Re-normalise prototypes (cosine)
        P0 = P0 / (np.linalg.norm(P0, axis=1, keepdims=True) + 1e-8)
        P1 = P1 / (np.linalg.norm(P1, axis=1, keepdims=True) + 1e-8)

        # cosine similarity == dot product after L2 norm
        logit0 = (Z_test @ P0.T) / temperature
        logit1 = (Z_test @ P1.T) / temperature
        logits = np.hstack([logit0, logit1])  # [n_test, 2]
        logits_accum += logits

    logits = logits_accum / float(repeats)
    # softmax to get p(ASD)
    exps = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = exps[:, 1] / (exps.sum(axis=1) + 1e-8)

    # metrics
    auroc = roc_auc_score(y_test, probs) if len(np.unique(y_test)) > 1 else np.nan
    auprc = average_precision_score(y_test, probs)
    y_pred = (probs >= 0.5).astype(int)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    return auroc, auprc, prec, rec

In [12]:
# ---- 3) Run participant-wise 3-fold CV with K-shot evaluation ----
for K in K_list:
    aucs, auprcs, precs, recs = [], [], [], []
    print(f"\n=== {K}-shot evaluation ===")
    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_participants, y_participants), 1):
        train_p = set(X_participants[tr_idx])
        test_p  = set(X_participants[te_idx])

        train_mask = np.array([pid in train_p for pid in IDs])
        test_mask  = np.array([pid in test_p for pid in IDs])

        Z_train, y_train = Z[train_mask], y[train_mask]
        Z_test,  y_test  = Z[test_mask],  y[test_mask]

        if Z_test.shape[0] == 0 or Z_train.shape[0] == 0:
            print(f"Fold {fold}: no data — skipped.")
            continue

        res = few_shot_fold_metrics(Z_train, y_train, Z_test, y_test, K)
        if res is None:
            print(f"Fold {fold}: missing a class in train — skipped.")
            continue

        auroc, auprc, prec, rec = res
        aucs.append(auroc); auprcs.append(auprc); precs.append(prec); recs.append(rec)
        auroc_txt = f"{auroc:.3f}" if not np.isnan(auroc) else "n/a"
        print(f"Fold {fold}: AUROC={auroc_txt} | AUPRC={auprc:.3f} | Precision={prec:.3f} | Recall={rec:.3f} "
              f"[train n={len(y_train)}, test n={len(y_test)}]")

    # summary for this K
    def mean_sd(a):
        a = np.array(a, dtype=float)
        a = a[~np.isnan(a)]
        return (np.mean(a), np.std(a)) if len(a) else (np.nan, np.nan)

    m_auc, s_auc = mean_sd(aucs)
    m_prc, s_prc = mean_sd(auprcs)
    m_p, s_p = mean_sd(precs)
    m_r, s_r = mean_sd(recs)
    auc_txt = f"{m_auc:.3f} ± {s_auc:.3f}" if not np.isnan(m_auc) else "n/a"
    print(f"== {K}-shot Summary ==  AUROC: {auc_txt} | AUPRC: {m_prc:.3f} ± {s_prc:.3f} | "
          f"Precision: {m_p:.3f} ± {s_p:.3f} | Recall: {m_r:.3f} ± {s_r:.3f}")



=== 1-shot evaluation ===
Fold 1: AUROC=0.871 | AUPRC=0.780 | Precision=0.417 | Recall=0.976 [train n=1836, test n=942]
Fold 2: AUROC=0.641 | AUPRC=0.578 | Precision=0.353 | Recall=0.862 [train n=1962, test n=816]
Fold 3: AUROC=0.785 | AUPRC=0.641 | Precision=0.366 | Recall=0.874 [train n=1758, test n=1020]
== 1-shot Summary ==  AUROC: 0.766 ± 0.095 | AUPRC: 0.666 ± 0.084 | Precision: 0.379 ± 0.028 | Recall: 0.904 ± 0.051

=== 3-shot evaluation ===
Fold 1: AUROC=0.870 | AUPRC=0.784 | Precision=0.462 | Recall=0.955 [train n=1836, test n=942]
Fold 2: AUROC=0.652 | AUPRC=0.585 | Precision=0.425 | Recall=0.678 [train n=1962, test n=816]
Fold 3: AUROC=0.828 | AUPRC=0.705 | Precision=0.683 | Recall=0.524 [train n=1758, test n=1020]
== 3-shot Summary ==  AUROC: 0.783 ± 0.095 | AUPRC: 0.692 ± 0.082 | Precision: 0.523 ± 0.114 | Recall: 0.719 ± 0.178

=== 5-shot evaluation ===
Fold 1: AUROC=0.879 | AUPRC=0.798 | Precision=0.485 | Recall=0.944 [train n=1836, test n=942]
Fold 2: AUROC=0.655 | AUP